In [ ]:
import os
from pathlib import Path

import polars as pl
import polars_st as st

In [ ]:
db_uri = os.environ["DATABASE_URL"]

In [ ]:
# Realtime ingestion config and cache toggle.
#
# Set this to True to reuse parquet snapshots.
# Set this to False to pull fresh data from Postgres and re-save parquet snapshots.
READ_FROM_PARQUET = True
WRITE_PARQUET_ON_DB_READ = True

ROUTES = ["4", "3", "A", "C"]
DIRECTION = 1  # MTA subway: 1 = southbound
WINDOW_DAYS = 14

PARQUET_DIR = Path("data") / "continuous_trajectories"
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

RT_TRIP_PARQUET = PARQUET_DIR / "realtime_trip_raw.parquet"
RT_STOP_TIME_PARQUET = PARQUET_DIR / "realtime_stop_time_raw.parquet"

if READ_FROM_PARQUET:
    missing = [p for p in [RT_TRIP_PARQUET, RT_STOP_TIME_PARQUET] if not p.exists()]
    if missing:
        missing_str = "\n".join(f"  - {p}" for p in missing)
        raise FileNotFoundError(
            "READ_FROM_PARQUET is True, but snapshot files are missing:\n"
            f"{missing_str}\n"
            "Set READ_FROM_PARQUET = False to pull from Postgres and create them."
        )

    df_rt_trip_raw = pl.read_parquet(RT_TRIP_PARQUET)
    df_rt_stop_time_raw = pl.read_parquet(RT_STOP_TIME_PARQUET)
    realtime_source = "parquet"
else:
    routes_literal = ",".join(f"'{r}'" for r in ROUTES)

    query_rt_trip = f"""
    SELECT id::text AS trip_id,
           source,
           route_id,
           direction,
           created_at
    FROM realtime.trip
    WHERE source = 'mta_subway'
      AND route_id IN ({routes_literal})
      AND direction = {DIRECTION}
      AND created_at >= NOW() - INTERVAL '{WINDOW_DAYS} days'
    """

    query_rt_stop_time = f"""
    WITH filtered_trips AS (
        SELECT id
        FROM realtime.trip
        WHERE source = 'mta_subway'
          AND route_id IN ({routes_literal})
          AND direction = {DIRECTION}
          AND created_at >= NOW() - INTERVAL '{WINDOW_DAYS} days'
    )
    SELECT st.trip_id::text AS trip_id,
           st.stop_id,
           st.arrival
    FROM realtime.stop_time st
    JOIN filtered_trips ft ON ft.id = st.trip_id
    ORDER BY st.trip_id, st.arrival
    """

    df_rt_trip_raw = pl.read_database_uri(query_rt_trip, db_uri)
    df_rt_stop_time_raw = pl.read_database_uri(query_rt_stop_time, db_uri)
    realtime_source = "database"

    if WRITE_PARQUET_ON_DB_READ:
        df_rt_trip_raw.write_parquet(RT_TRIP_PARQUET)
        df_rt_stop_time_raw.write_parquet(RT_STOP_TIME_PARQUET)

df_rt = (
    df_rt_stop_time_raw.join(df_rt_trip_raw, on="trip_id", how="inner")
    .select(["trip_id", "route_id", "direction", "created_at", "stop_id", "arrival"])
    .sort(["trip_id", "arrival"])
)

print(f"Realtime source: {realtime_source}")
print(
    f"Loaded {df_rt.height} stop-time rows across "
    f"{df_rt['trip_id'].n_unique()} trips "
    f"on routes {ROUTES} direction={DIRECTION} over the last {WINDOW_DAYS} days"
)
df_rt.head()

In [ ]:
pl.Config.set_tbl_rows(100)
df_rt.filter(trip_id="019d87a5-e1a3-7c63-aaa2-9020f40411f1")

In [ ]:
query = """
SELECT id,
       source,
       long_name,
       short_name,
       color,
       data,
       ST_AsEWKB(geom) AS geometry
FROM static.route
WHERE source = 'mta_subway' AND geom IS NOT NULL
"""
# pl.read_database_uri(query, db_uri).write_parquet(PARQUET_DIR / "routes.parquet")

df_routes = pl.read_parquet(PARQUET_DIR / "routes.parquet")

In [ ]:
query = """
SELECT id,
       source,
       name,
       ST_AsEWKB(geom) AS geometry,
       data
FROM static.stop
WHERE source = 'mta_subway' AND geom IS NOT NULL
"""

# pl.read_database_uri(query, db_uri).write_parquet(PARQUET_DIR / "stops.parquet")
df_stops = pl.read_parquet(PARQUET_DIR / "stops.parquet")

In [ ]:
query = """
SELECT route_id,
       stop_id,
       source,
       stop_sequence,
       data
FROM static.route_stop
WHERE source = 'mta_subway'
    AND COALESCE(data->>'stop_type', '') IN ('full_time', 'part_time')
"""

# pl.read_database_uri(query, db_uri).write_parquet(PARQUET_DIR / "route_stops.parquet")
df_route_stops = pl.read_parquet(PARQUET_DIR / "route_stops.parquet")

In [ ]:
# 1. Join stops to route_stops to get the stop geometries in order
df_stops_ordered = df_route_stops.join(
    df_stops.select(["id", "geometry"]), left_on="stop_id", right_on="id", how="left"
).rename({"geometry": "stop_geom"})  # Rename to avoid conflict with route geometry

# Sort by route and stop sequence to ensure they are in the correct travel order
df_stops_ordered = df_stops_ordered.sort(["route_id", "stop_sequence"])

# 2. Join the ordered stops with the route geometries
df_joined = df_stops_ordered.join(
    df_routes.select(["id", "geometry"]), left_on="route_id", right_on="id", how="left"
).rename({"geometry": "route_geom"})

# 3. Project stops onto the route line to get fractional distances
# st.project with normalized=True returns a float between 0.0 and 1.0 representing how far along the line the point is
df_projected = df_joined.with_columns(
    pl.col("route_geom")
    .st.project(pl.col("stop_geom"), normalized=True)
    .alias("fraction_start")
)

# 4. Use Polars window functions to get the destination stop for each segment
df_segments = df_projected.with_columns(
    pl.col("fraction_start").shift(-1).over("route_id").alias("fraction_end"),
    pl.col("stop_id").shift(-1).over("route_id").alias("next_stop_id"),
)

# Drop the last stop of each route, as it doesn't have a "next" stop to form a segment
df_segments = df_segments.drop_nulls(subset=["fraction_end"])

# 5. Extract the geometry between the start and end fractions
df_segments = df_segments.with_columns(
    pl.col("route_geom")
    .st.substring(pl.col("fraction_start"), pl.col("fraction_end"))
    .alias("segment_geom")
)

# df_final_segments = df_segments.select(
#     ["route_id", "stop_sequence", "stop_id", "next_stop_id", "segment_geom"]
# )

In [ ]:
import heapq
from collections import defaultdict
import datetime as dt

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import (
    CubicSpline,
    PchipInterpolator,
    Akima1DInterpolator,
    UnivariateSpline,
)
from scipy.optimize import brentq


def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    r = 6371008.8  # mean Earth radius in meters
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2.0) ** 2
    return float(2.0 * r * np.arcsin(np.sqrt(a)))


# Build stop coordinate lookup (WGS84 lon/lat from point geometry).
stop_coords_df = (
    df_stops.select(["id", "geometry"])
    .with_columns(
        lon=pl.col("geometry").st.x(),
        lat=pl.col("geometry").st.y(),
    )
    .select(["id", "lat", "lon"])
    .drop_nulls(["lat", "lon"])
    .unique(subset=["id"], keep="first")
)
stop_coord_map = {
    row["id"]: (float(row["lat"]), float(row["lon"]))
    for row in stop_coords_df.iter_rows(named=True)
}

# Build per-route ordered stop list from static route_stop.
route_stop_rows = (
    df_route_stops.filter(pl.col("route_id").is_in(ROUTES))
    .select(["route_id", "stop_id", "stop_sequence"])
    .sort(["route_id", "stop_sequence"])
)

route_order_rows: dict[str, list[tuple[int, str]]] = defaultdict(list)
for row in route_stop_rows.iter_rows(named=True):
    sid = row["stop_id"]
    if sid not in stop_coord_map:
        continue
    route_order_rows[row["route_id"]].append((int(row["stop_sequence"]), sid))

# Build an undirected weighted graph of adjacent route_stop pairs for each route.
route_graph: dict[str, dict[str, dict[str, float]]] = {}
for route_id, seq_rows in route_order_rows.items():
    adj: dict[str, dict[str, float]] = defaultdict(dict)
    for (_, u), (_, v) in zip(seq_rows, seq_rows[1:]):
        if u == v:
            continue
        lat1, lon1 = stop_coord_map[u]
        lat2, lon2 = stop_coord_map[v]
        w = haversine_m(lat1, lon1, lat2, lon2)
        prev = adj[u].get(v)
        if prev is None or w < prev:
            adj[u][v] = w
            adj[v][u] = w
    route_graph[route_id] = {k: dict(v) for k, v in adj.items()}

_shortest_path_cache: dict[tuple[str, str, str], float | None] = {}


def shortest_path_distance(
    route_id: str, start_stop: str, end_stop: str
) -> float | None:
    """Shortest-path distance in meters on the route stop graph."""
    if start_stop == end_stop:
        return 0.0
    key = (route_id, start_stop, end_stop)
    if key in _shortest_path_cache:
        return _shortest_path_cache[key]

    g = route_graph.get(route_id, {})
    if start_stop not in g or end_stop not in g:
        _shortest_path_cache[key] = None
        return None

    dist: dict[str, float] = {start_stop: 0.0}
    pq: list[tuple[float, str]] = [(0.0, start_stop)]

    while pq:
        d, u = heapq.heappop(pq)
        if u == end_stop:
            _shortest_path_cache[key] = d
            _shortest_path_cache[(route_id, end_stop, start_stop)] = d
            return d
        if d > dist.get(u, float("inf")):
            continue
        for v, w in g.get(u, {}).items():
            nd = d + w
            if nd < dist.get(v, float("inf")):
                dist[v] = nd
                heapq.heappush(pq, (nd, v))

    _shortest_path_cache[key] = None
    return None


# Build a monotone static stop-distance table for diagnostics and joins.
# Distance is cumulative shortest-path along stop_sequence order.
rows: list[dict] = []
for route_id, seq_rows in route_order_rows.items():
    seen: set[str] = set()
    route_rows: list[dict] = []
    prev_stop: str | None = None
    cum_m = 0.0

    for stop_sequence, stop_id in seq_rows:
        if stop_id in seen:
            continue
        seen.add(stop_id)

        if prev_stop is not None:
            d = shortest_path_distance(route_id, prev_stop, stop_id)
            if d is None:
                lat1, lon1 = stop_coord_map[prev_stop]
                lat2, lon2 = stop_coord_map[stop_id]
                d = haversine_m(lat1, lon1, lat2, lon2)
            cum_m += float(d)

        route_rows.append(
            {
                "route_id": route_id,
                "stop_id": stop_id,
                "stop_sequence": int(stop_sequence),
                "cum_distance_m": float(cum_m),
            }
        )
        prev_stop = stop_id

    route_length_m = float(route_rows[-1]["cum_distance_m"]) if route_rows else 0.0
    for r in route_rows:
        r["route_length_m"] = route_length_m
        r["fraction"] = (
            r["cum_distance_m"] / route_length_m if route_length_m > 0 else 0.0
        )
    rows.extend(route_rows)

df_stops_with_dist = pl.DataFrame(rows).sort(["route_id", "stop_sequence"])

route_graph_summary = []
for route_id in ROUTES:
    g = route_graph.get(route_id, {})
    n_nodes = len(g)
    n_edges = sum(len(vs) for vs in g.values()) // 2
    route_len = (
        float(
            df_stops_with_dist.filter(pl.col("route_id") == route_id)[
                "route_length_m"
            ].max()  # pyright: ignore[reportArgumentType]
        )
        if df_stops_with_dist.filter(pl.col("route_id") == route_id).height > 0
        else 0.0
    )
    route_graph_summary.append(
        {
            "route_id": route_id,
            "graph_nodes": n_nodes,
            "graph_edges": n_edges,
            "graph_route_length_km": route_len / 1000.0,
        }
    )

print(
    f"Graph-based stop distance table: {df_stops_with_dist.height} rows across "
    f"{df_stops_with_dist['route_id'].n_unique()} routes"
)
display(pl.DataFrame(route_graph_summary).sort("route_id"))
df_stops_with_dist.head(10)

In [ ]:
df_stops_with_dist.filter(pl.col("route_id") == "4").sort("route_id")

In [ ]:
# Geometry sanity: how many distinct projected distances exist per route?
# If a route has many stops but very few distinct cum_distance_m values,
# distance-dedupe will collapse trips to a tiny stop count.

route_geom_quality = (
    df_stops_with_dist.group_by("route_id")
    .agg(
        n_rows=pl.len(),
        n_stops=pl.col("stop_id").n_unique(),
        n_unique_distance=pl.col("cum_distance_m").n_unique(),
        n_unique_distance_round1=pl.col("cum_distance_m").round(1).n_unique(),
        n_unique_distance_round0=pl.col("cum_distance_m").round(0).n_unique(),
    )
    .with_columns(
        uniq_ratio=(pl.col("n_unique_distance") / pl.col("n_stops")),
        uniq_ratio_r1=(pl.col("n_unique_distance_round1") / pl.col("n_stops")),
    )
    .sort("route_id")
)

display(route_geom_quality.sort("uniq_ratio"))

# print()
# print("=== Route projected distance frequencies (top 12) ===")
# display(
#     df_stops_with_dist.filter(pl.col("route_id") == "4")
#     .group_by(pl.col("cum_distance_m").round(1).alias("s_round1_m"))
#     .agg(n=pl.len(), sample_stops=pl.col("stop_id").head(5))
#     .sort("n", descending=True)
#     .head(12)
# )


In [ ]:
print(f"Realtime source in use: {realtime_source}")
print(
    f"Trip rows: {df_rt_trip_raw.height:,} | "
    f"Stop-time rows: {df_rt_stop_time_raw.height:,} | "
    f"Joined rows: {df_rt.height:,}"
)
print(
    f"Trips: {df_rt['trip_id'].n_unique():,} | "
    f"Routes: {df_rt['route_id'].n_unique():,} | "
    f"Window: {WINDOW_DAYS} day(s)"
)
df_rt.head()

In [ ]:
# Attach cumulative distance to each arrival, group into per-trip arrays,
# and filter to trips that are usable for cubic-spline fitting.

# Requirements for a usable trip:

# - at least 4 stops (CubicSpline needs >= 4 knots for a meaningful fit)
# - strictly monotone arrival timestamps (time must be a function of stop)
# - monotone cumulative distance (train only moves forward along the route).
#   The stored route LineString may be oriented either way relative to the
#   direction of travel, so a trip with strictly _decreasing_ s is equally
#   valid: we simply flip it to strictly increasing via s' = L - s.

# Small tolerances handle realtime-feed quirks where two consecutive stops
# share the same predicted arrival second, or where st.project rounding
# puts two stops at near-identical fractions.

TIME_TOL_S = 0.5  # seconds; dedupe arrivals within this window
DIST_TOL_M = 1.0  # meters; dedupe stops within this much of each other


def _classify_monotone(xs: list[float], tol: float) -> str:
    """Return 'inc', 'dec', or 'none' after a tol-tolerant dedupe check."""
    diffs = np.diff(xs)
    if np.all(diffs > -tol) and np.any(diffs > tol):
        return "inc"
    if np.all(diffs < tol) and np.any(diffs < -tol):
        return "dec"
    return "none"


# Build per-trip cumulative distance directly from stop-to-stop shortest paths
# on the route stop graph (robust to skipped stops and branch detours).
route_length_lookup = {
    row["route_id"]: float(row["route_length_m"])
    for row in (
        df_stops_with_dist.group_by("route_id")
        .agg(pl.col("route_length_m").max().alias("route_length_m"))
        .iter_rows(named=True)
    )
}

df_trips_input = (
    df_rt.filter(pl.col("route_id").is_in(ROUTES), pl.col("direction") == DIRECTION)
    .select(["trip_id", "route_id", "direction", "created_at", "stop_id", "arrival"])
    .sort(["trip_id", "arrival"])
)

grouped_rt = df_trips_input.group_by(["trip_id", "route_id", "direction"]).agg(
    pl.col("created_at"),
    pl.col("arrival"),
    pl.col("stop_id"),
)

rt_geo_rows: list[dict] = []
trip_graph_gap_rows: list[dict] = []

for row in grouped_rt.iter_rows(named=True):
    trip_id = row["trip_id"]
    route_id = row["route_id"]
    direction = row["direction"]

    created_at = list(row["created_at"])
    arrivals = list(row["arrival"])
    stop_ids = list(row["stop_id"])

    order = sorted(range(len(arrivals)), key=lambda i: arrivals[i])
    created_at = [created_at[i] for i in order]
    arrivals = [arrivals[i] for i in order]
    stop_ids = [stop_ids[i] for i in order]

    if not arrivals:
        continue

    cum_s: list[float] = [0.0]
    n_pairs = 0
    n_missing_paths = 0

    for i in range(1, len(stop_ids)):
        prev_stop = stop_ids[i - 1]
        curr_stop = stop_ids[i]
        n_pairs += 1

        d = shortest_path_distance(route_id, prev_stop, curr_stop)
        if d is None:
            n_missing_paths += 1
            if prev_stop in stop_coord_map and curr_stop in stop_coord_map:
                lat1, lon1 = stop_coord_map[prev_stop]
                lat2, lon2 = stop_coord_map[curr_stop]
                d = haversine_m(lat1, lon1, lat2, lon2)
            else:
                d = 0.0

        cum_s.append(cum_s[-1] + float(d))

    route_length_m = route_length_lookup.get(route_id, max(cum_s[-1], 1.0))
    if route_length_m <= 0:
        route_length_m = max(cum_s[-1], 1.0)

    trip_graph_gap_rows.append(
        {
            "trip_id": trip_id,
            "route_id": route_id,
            "n_pairs": n_pairs,
            "n_missing_paths": n_missing_paths,
        }
    )

    for i in range(len(arrivals)):
        rt_geo_rows.append(
            {
                "trip_id": trip_id,
                "route_id": route_id,
                "direction": direction,
                "created_at": created_at[i],
                "stop_id": stop_ids[i],
                "arrival": arrivals[i],
                "cum_distance_m": float(cum_s[i]),
                "route_length_m": float(route_length_m),
            }
        )

df_rt_geo = pl.DataFrame(rt_geo_rows).sort(["trip_id", "arrival"])

df_trips_grouped = (
    df_rt_geo.group_by(["trip_id", "route_id", "direction"])
    .agg(
        pl.col("arrival"),
        pl.col("cum_distance_m"),
        pl.col("stop_id"),
        pl.col("route_length_m").first().alias("route_length_m"),
    )
    .with_columns(n_stops=pl.col("arrival").list.len())
)

trip_graph_gap_df = pl.DataFrame(trip_graph_gap_rows)
gap_summary = (
    trip_graph_gap_df.select(
        pl.len().alias("n_trips"),
        pl.col("n_missing_paths").sum().alias("total_missing_paths"),
        ((pl.col("n_missing_paths") > 0).sum()).alias("trips_with_any_missing_path"),
    )
    if trip_graph_gap_df.height > 0
    else pl.DataFrame(
        [{"n_trips": 0, "total_missing_paths": 0, "trips_with_any_missing_path": 0}]
    )
)

valid_trips: list[dict] = []
n_total = df_trips_grouped.height
n_too_short = 0
n_nonmono_time = 0
n_nonmono_dist = 0
n_flipped = 0
example_bad: dict | None = None

# Dedupe diagnostics: count stop-time points removed by each tolerance stage.
raw_stop_times_total = 0
after_time_stop_times_total = 0
dist_stage_input_total = 0
after_dist_stop_times_total = 0
time_dedupe_removed_total = 0
dist_dedupe_removed_total = 0
dedupe_by_route: dict[str, dict[str, int]] = {}

for row in df_trips_grouped.iter_rows(named=True):
    arrivals = list(row["arrival"])
    s_m = list(row["cum_distance_m"])
    stop_ids = list(row["stop_id"])
    route_id = row["route_id"]
    L = float(row["route_length_m"])

    route_stats = dedupe_by_route.setdefault(
        route_id,
        {
            "trips": 0,
            "raw": 0,
            "time_removed": 0,
            "after_time": 0,
            "dist_stage_input": 0,
            "dist_removed": 0,
            "after_dist": 0,
        },
    )
    route_stats["trips"] += 1

    # Sort by arrival (should already be sorted) and dedupe ties within TIME_TOL_S.
    order = sorted(range(len(arrivals)), key=lambda i: arrivals[i])
    arrivals = [arrivals[i] for i in order]
    s_m = [s_m[i] for i in order]
    stop_ids = [stop_ids[i] for i in order]

    n_raw = len(arrivals)
    raw_stop_times_total += n_raw
    route_stats["raw"] += n_raw

    a_sec = [(a - arrivals[0]).total_seconds() for a in arrivals]
    keep_idx = [0]
    for i in range(1, len(a_sec)):
        if a_sec[i] - a_sec[keep_idx[-1]] > TIME_TOL_S:
            keep_idx.append(i)
    arrivals = [arrivals[i] for i in keep_idx]
    s_m = [s_m[i] for i in keep_idx]
    stop_ids = [stop_ids[i] for i in keep_idx]

    n_after_time = len(arrivals)
    time_removed = n_raw - n_after_time
    time_dedupe_removed_total += time_removed
    after_time_stop_times_total += n_after_time
    route_stats["time_removed"] += time_removed
    route_stats["after_time"] += n_after_time

    n = n_after_time
    if n < 4:
        n_too_short += 1
        continue

    # Time must now be strictly increasing after the tol dedupe.
    if not all(arrivals[i] > arrivals[i - 1] for i in range(1, n)):
        n_nonmono_time += 1
        if example_bad is None:
            example_bad = {"reason": "time", "arrivals": arrivals[:6], "s_m": s_m[:6]}
        continue

    # Accept either strictly-increasing or strictly-decreasing distance.
    kind = _classify_monotone(s_m, DIST_TOL_M)
    if kind == "none":
        n_nonmono_dist += 1
        if example_bad is None:
            example_bad = {
                "reason": "dist",
                "stop_ids": stop_ids[:8],
                "s_m": [round(x, 1) for x in s_m[:8]],
            }
        continue
    if kind == "dec":
        s_m = [L - x for x in s_m]
        n_flipped += 1

    # After a potential flip the sequence should be (approximately) increasing;
    # enforce strict monotonicity by dropping any tol-ties that remain.
    n_before_dist = len(s_m)
    keep = [0]
    for i in range(1, len(s_m)):
        if s_m[i] - s_m[keep[-1]] > DIST_TOL_M:
            keep.append(i)
    arrivals = [arrivals[i] for i in keep]
    s_m = [s_m[i] for i in keep]
    stop_ids = [stop_ids[i] for i in keep]

    n_after_dist = len(arrivals)
    dist_removed = n_before_dist - n_after_dist
    dist_stage_input_total += n_before_dist
    dist_dedupe_removed_total += dist_removed
    after_dist_stop_times_total += n_after_dist
    route_stats["dist_stage_input"] += n_before_dist
    route_stats["dist_removed"] += dist_removed
    route_stats["after_dist"] += n_after_dist

    n = n_after_dist
    if n < 4:
        n_too_short += 1
        continue

    valid_trips.append(
        {
            "trip_id": row["trip_id"],
            "route_id": row["route_id"],
            "direction": row["direction"],
            "arrivals": arrivals,
            "s_m": np.asarray(s_m, dtype=float),
            "stop_ids": stop_ids,
            "route_length_m": L,
            "n_stops": n,
            "reversed": kind == "dec",
        }
    )

print("=== Graph distance construction ===")
print(
    f"Trips built: {int(gap_summary['n_trips'][0]):,} | "
    f"Trips with missing path links: {int(gap_summary['trips_with_any_missing_path'][0]):,} | "
    f"Total missing links: {int(gap_summary['total_missing_paths'][0]):,}"
)

print()
print("=== Dedupe diagnostics ===")
print(f"Thresholds -> TIME_TOL_S={TIME_TOL_S}s, DIST_TOL_M={DIST_TOL_M}m")
time_removed_pct = (
    100.0 * time_dedupe_removed_total / raw_stop_times_total
    if raw_stop_times_total
    else 0.0
)
dist_removed_pct = (
    100.0 * dist_dedupe_removed_total / dist_stage_input_total
    if dist_stage_input_total
    else 0.0
)
print(
    f"Time dedupe removed: {time_dedupe_removed_total:,} / {raw_stop_times_total:,} "
    f"({time_removed_pct:.2f}%) stop-times"
)
print(
    f"Distance dedupe removed: {dist_dedupe_removed_total:,} / {dist_stage_input_total:,} "
    f"({dist_removed_pct:.2f}%) stop-times entering distance stage"
)
print(
    f"After time dedupe: {after_time_stop_times_total:,} | "
    f"After distance dedupe: {after_dist_stop_times_total:,}"
)

dedupe_route_rows = []
for route_id, stats in dedupe_by_route.items():
    raw = stats["raw"]
    dist_in = stats["dist_stage_input"]
    dedupe_route_rows.append(
        {
            "route_id": route_id,
            "trips": stats["trips"],
            "raw_stop_times": raw,
            "time_removed": stats["time_removed"],
            "time_removed_pct": (100.0 * stats["time_removed"] / raw) if raw else 0.0,
            "dist_stage_input": dist_in,
            "dist_removed": stats["dist_removed"],
            "dist_removed_pct": (100.0 * stats["dist_removed"] / dist_in)
            if dist_in
            else 0.0,
            "after_dist": stats["after_dist"],
        }
    )

if dedupe_route_rows:
    display(pl.DataFrame(dedupe_route_rows).sort("route_id"))

print()
print(f"Total trips considered:              {n_total}")
print(f"Dropped - fewer than 4 stops:         {n_too_short}")
print(f"Dropped - non-monotone arrival times: {n_nonmono_time}")
print(f"Dropped - non-monotone distances:     {n_nonmono_dist}")
print(f"Flipped (decreasing -> increasing):   {n_flipped}")
print(f"Valid trips kept:                     {len(valid_trips)}")
if example_bad is not None and len(valid_trips) == 0:
    print()
    print("Example trip that was dropped:")
    print(example_bad)
if valid_trips:
    _n_stops = [t["n_stops"] for t in valid_trips]
    print(
        f"Stops per trip: min={min(_n_stops)} median={int(np.median(_n_stops))} "
        f"max={max(_n_stops)}"
    )

In [ ]:
# Per-trip fitters factory
# We will compare multiple methods:
# - Cubic Spline (Clamped & Natural)
# - PCHIP (Monotonic)
# - Akima (Stable to outliers)
# - UnivariateSpline (Exact interpolation s=0, and Smoothed s=10)
# - Linear


def trip_time_seconds(arrivals: list[dt.datetime]) -> np.ndarray:
    """Convert list of timestamps to seconds since the first arrival."""
    t0 = arrivals[0]
    return np.asarray([(a - t0).total_seconds() for a in arrivals], dtype=float)


def get_interpolator(method: str, t_seconds: np.ndarray, s_meters: np.ndarray):
    """Returns a callable interpolant and a string name."""
    if method == "cubic_clamped":
        return CubicSpline(
            t_seconds, s_meters, bc_type=((1, 0.0), (1, 0.0)), extrapolate=False
        )
    elif method == "cubic_natural":
        return CubicSpline(t_seconds, s_meters, bc_type="natural", extrapolate=False)
    elif method == "pchip":
        return PchipInterpolator(t_seconds, s_meters, extrapolate=False)
    elif method == "akima":
        return Akima1DInterpolator(t_seconds, s_meters)
    elif method == "univariate_s0":
        return UnivariateSpline(t_seconds, s_meters, s=0, ext=2)
    elif method == "univariate_smoothed":
        # s depends on number of points, we use a simple smoothing heuristic
        return UnivariateSpline(t_seconds, s_meters, s=len(t_seconds) * 10, ext=2)
    elif method == "linear":

        def f(t):
            return np.interp(t, t_seconds, s_meters)

        return f
    else:
        raise ValueError(f"Unknown method {method}")


METHODS_TO_COMPARE = [
    "cubic_clamped",
    "cubic_natural",
    "pchip",
    "akima",
    "univariate_s0",
    "univariate_smoothed",
    "linear",
]

# Demo: fit the longest trip we have and print a quick summary.
demo_trip = max(valid_trips, key=lambda x: x["n_stops"])
demo_t = trip_time_seconds(demo_trip["arrivals"])
demo_s = demo_trip["s_m"]

print(f"Demo trip_id={demo_trip['trip_id']} route={demo_trip['route_id']}")
print(
    f"  {demo_trip['n_stops']} stops, duration={demo_t[-1] / 60:.1f} min, "
    f"distance covered={demo_s[-1] / 1000:.2f} km "
    f"(route length={demo_trip['route_length_m'] / 1000:.2f} km)"
)
print(
    f"  Avg speed = {demo_s[-1] / demo_t[-1]:.2f} m/s "
    f"({demo_s[-1] / demo_t[-1] * 2.237:.1f} mph)"
)

In [ ]:
# Physical plausibility thresholds for NYC subway:
#   - |v| should not exceed ~35 m/s (~78 mph). R142 top speed is ~55 mph.
#   - |a| should not exceed ~1.5 m/s^2 (service acceleration/brake is ~1.0 m/s^2).

SANITY_V_MAX = 35.0  # m/s
SANITY_A_MAX = 1.5  # m/s^2

method_physics_stats = []
implausible_trips_by_method = {m: [] for m in METHODS_TO_COMPARE}

for method in METHODS_TO_COMPARE:
    flagged = 0
    v_maxes = []
    a_maxes = []

    for trip in valid_trips:
        arrivals = list(trip["arrivals"])
        s_m = np.asarray(trip["s_m"], dtype=float)
        t_sec = trip_time_seconds(arrivals)

        try:
            interp = get_interpolator(method, t_sec, s_m)

            # Sample at 500 points across the trip
            ts = np.linspace(t_sec[0], t_sec[-1], 500)
            s_vals = interp(ts)

            # Use numerical differentiation if analytic derivative is not available
            if hasattr(interp, "derivative"):
                v = interp.derivative(1)(ts)
                a = interp.derivative(2)(ts)
            else:
                dt_step = ts[1] - ts[0]
                v = np.gradient(s_vals, dt_step)
                a = np.gradient(v, dt_step)

            vmax = float(np.nanmax(np.abs(v)))
            amax = float(np.nanmax(np.abs(a)))

            v_maxes.append(vmax)
            a_maxes.append(amax)

            if vmax > SANITY_V_MAX or amax > SANITY_A_MAX:
                flagged += 1
                implausible_trips_by_method[method].append(trip["trip_id"])
        except Exception:
            # Fit failed
            pass

    method_physics_stats.append(
        {
            "Method": method,
            "Flagged Trips": flagged,
            "% Flagged": round(flagged / len(valid_trips) * 100, 2)
            if valid_trips
            else 0,
            "Median V_max (m/s)": round(np.median(v_maxes), 2) if v_maxes else 0,
            "95th Pct A_max (m/s^2)": round(np.percentile(a_maxes, 95), 3)
            if a_maxes
            else 0,
        }
    )

df_physics = pl.DataFrame(method_physics_stats)
print("=== Physics Plausibility by Method ===")
display(df_physics)

# Visualization of a bad trip
if implausible_trips_by_method["cubic_clamped"]:
    bad_trip_id = implausible_trips_by_method["cubic_clamped"][0]
    bad_trip = next((t for t in valid_trips if t["trip_id"] == bad_trip_id), None)

    if bad_trip:
        t_sec = trip_time_seconds(bad_trip["arrivals"])
        s_m = np.asarray(bad_trip["s_m"], dtype=float)
        ts_dense = np.linspace(t_sec[0], t_sec[-1], 500)

        fig, axs = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
        axs[0].plot(t_sec, s_m, "ko", label="Data (Stops)")
        axs[0].set_ylabel("Distance (m)")
        axs[0].set_title(f"Trip {bad_trip_id} (Flagged by Cubic Clamped)")

        for method in ["cubic_clamped", "pchip", "univariate_smoothed"]:
            interp = get_interpolator(method, t_sec, s_m)
            s_dense = interp(ts_dense)

            if hasattr(interp, "derivative"):
                v_dense = interp.derivative(1)(ts_dense)
                a_dense = interp.derivative(2)(ts_dense)
            else:
                dt_step = ts_dense[1] - ts_dense[0]
                v_dense = np.gradient(s_dense, dt_step)
                a_dense = np.gradient(v_dense, dt_step)

            axs[0].plot(ts_dense, s_dense, label=method)
            axs[1].plot(ts_dense, v_dense, label=method)
            axs[2].plot(ts_dense, a_dense, label=method)

        axs[1].axhline(SANITY_V_MAX, color="r", linestyle="--", label="Sanity V_max")
        axs[1].set_ylabel("Velocity (m/s)")

        axs[2].axhline(SANITY_A_MAX, color="r", linestyle="--", label="Sanity A_max")
        axs[2].set_ylabel("Acceleration (m/s^2)")
        axs[2].set_xlabel("Time (s)")

        for ax in axs:
            ax.legend()
            ax.grid(True)

        plt.tight_layout()
        plt.show()

In [ ]:
df_valid_trips = (
    pl.from_dicts(valid_trips)
    .explode("arrivals", "s_m", "stop_ids")
    .rename({"arrivals": "arrival", "s_m": "cum_distance_m", "stop_ids": "stop_id"})
)

In [ ]:
# (
#     df_valid_trips.filter(pl.col("trip_id") == flagged_trips[10])
#     .sort("arrival")
#     .with_columns(
#         next_arrival=pl.col("arrival").shift(-1),
#         arrival_diff_s=(
#             pl.col("arrival").shift(-1) - pl.col("arrival")
#         ).dt.total_seconds(),
#     )
# )

In [ ]:
# Leave-one-out cross-validation
def cv_errors(trips: list[dict], method: str):
    """Return (errors, per_stop) where per_stop is (relative_position, err_s)."""
    errors: list[float] = []
    failures = 0

    for trip in trips:
        t_sec = trip_time_seconds(trip["arrivals"])
        s_m = trip["s_m"]
        n = len(t_sec)

        if n < 5:
            continue

        for i in range(1, n - 1):
            mask = np.ones(n, dtype=bool)
            mask[i] = False
            t_train = t_sec[mask]
            s_train = s_m[mask]

            try:
                f = get_interpolator(method, t_train, s_train)
                s_target = float(s_m[i])

                def g(t):
                    return float(f(t)) - s_target

                t_lo, t_hi = float(t_sec[i - 1]), float(t_sec[i + 1])
                g_lo, g_hi = g(t_lo), g(t_hi)

                if np.isnan(g_lo) or np.isnan(g_hi) or g_lo * g_hi > 0:
                    failures += 1
                    continue

                t_pred = brentq(g, t_lo, t_hi)
                err = t_pred - float(t_sec[i])
                errors.append(err)
            except Exception:
                failures += 1
                continue

    return np.asarray(errors), failures


cv_stats = []
for method in METHODS_TO_COMPARE:
    errs, fails = cv_errors(valid_trips, method=method)
    if len(errs) > 0:
        cv_stats.append(
            {
                "Method": method,
                "CV Predictions": len(errs),
                "Skipped/Failed": fails,
                "MAE (s)": round(float(np.mean(np.abs(errs))), 2),
                "RMSE (s)": round(float(np.sqrt(np.mean(errs**2))), 2),
                "Median Bias (s)": round(float(np.median(errs)), 2),
                "Max |Err| (s)": round(float(np.max(np.abs(errs))), 2),
            }
        )

df_cv = pl.DataFrame(cv_stats)
print("=== Cross-Validation Errors by Method ===")
display(df_cv)

In [ ]:
# Visualization 1: position, velocity, and acceleration over time for a
# representative trip. Linear interpolation only gives a position curve;
# its "velocity" is a step function (derivative of linear segments) and its
# "acceleration" is zero everywhere except at knots, which is exactly why
# we need the cubic spline for physically meaningful v and a.

trip = demo_trip  # longest trip, selected earlier
t_sec = trip_time_seconds(trip["arrivals"])
s_m = trip["s_m"]

spline = fit_cubic(t_sec, s_m, clamped=True)
spline_nat = fit_cubic(t_sec, s_m, clamped=False)
v_fn = spline.derivative(1)
a_fn = spline.derivative(2)

ts = np.linspace(t_sec[0], t_sec[-1], 2000)

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

axes[0].plot(ts / 60, spline(ts) / 1000, label="Cubic (clamped)", color="tab:blue")
axes[0].plot(
    ts / 60, spline_nat(ts) / 1000, ":", label="Cubic (natural)", color="tab:cyan"
)
axes[0].plot(
    ts / 60,
    np.interp(ts, t_sec, s_m) / 1000,
    "--",
    label="Linear baseline",
    color="tab:orange",
)
axes[0].scatter(
    t_sec / 60,
    s_m / 1000,
    color="red",
    zorder=5,
    s=30,
    label="Observed arrivals",
)
axes[0].set_ylabel("Distance along route (km)")
axes[0].set_title(
    f"Trip {trip['trip_id'][:8]}..  route {trip['route_id']}  "
    f"({trip['n_stops']} stops, {t_sec[-1] / 60:.1f} min)"
)
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.3)

axes[1].plot(ts / 60, v_fn(ts), color="tab:blue")
axes[1].axhline(0, color="k", lw=0.5)
axes[1].axhline(
    SANITY_V_MAX, color="red", lw=0.5, ls=":", label=f"|v| = {SANITY_V_MAX}"
)
axes[1].axhline(-SANITY_V_MAX, color="red", lw=0.5, ls=":")
axes[1].set_ylabel("Velocity s'(t)  (m/s)")
axes[1].legend(loc="upper right")
axes[1].grid(alpha=0.3)

axes[2].plot(ts / 60, a_fn(ts), color="tab:blue")
axes[2].axhline(0, color="k", lw=0.5)
axes[2].axhline(
    SANITY_A_MAX, color="red", lw=0.5, ls=":", label=f"|a| = {SANITY_A_MAX}"
)
axes[2].axhline(-SANITY_A_MAX, color="red", lw=0.5, ls=":")
axes[2].set_ylabel("Acceleration s''(t)  (m/s^2)")
axes[2].set_xlabel("Time since first stop (min)")
axes[2].legend(loc="upper right")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: map the interpolated train position for the demo trip.
#
# Pipeline:
#   1. At each elapsed time t, evaluate the cubic spline to get s(t) in meters.
#   2. Normalize by the route length to get a fraction f in [0, 1]. If cell 9
#      flipped this trip (stored LineString runs opposite to travel), un-flip
#      to 1 - f so we index the stored line correctly.
#   3. Use polars-st's st.interpolate(f, normalized=True) to resolve the
#      lon/lat point at fractional distance f along the route LineString.
# This is the continuous-position function promised by the proposal.

route_wkb = df_routes.filter(pl.col("id") == trip["route_id"])["geometry"].to_list()[0]
reversed_trip = bool(trip.get("reversed", False))

# Sample N_FRAMES evenly-spaced times and turn each into a fraction along the
# stored route line.
N_FRAMES = 30
sample_times = np.linspace(t_sec[0], t_sec[-1], N_FRAMES)
sample_s = spline(sample_times)
sample_frac = np.clip(sample_s / trip["route_length_m"], 0.0, 1.0)
if reversed_trip:
    sample_frac = 1.0 - sample_frac

points_df = (
    pl.DataFrame(
        {
            "t_sec": sample_times,
            "fraction": sample_frac,
            "route_wkb": [route_wkb] * len(sample_times),
        }
    )
    # TODO: why is this getting converted to wkb in the first place?
    .with_columns(route_geom=st.from_wkb("route_wkb"))
    .with_columns(
        geom=pl.col("route_geom").st.interpolate(pl.col("fraction"), normalized=True),
        label=pl.format("train_t={}min", (pl.col("t_sec") / 60).round(1)),
    )
    .select(["label", "geom"])
)

route_df = (
    pl.DataFrame({"route_wkb": [route_wkb]})
    .with_columns(
        label=pl.lit("route"),
        geom=st.from_wkb("route_wkb"),
    )
    .select(["label", "geom"])
)

viz_df = pl.concat([route_df, points_df], how="vertical")

# "Current" snapshot at 40% through the trip for the printed summary.
elapsed_now = float((t_sec[-1] - t_sec[0]) * 0.4)
s_now = float(spline(elapsed_now))
frac_now = float(np.clip(s_now / trip["route_length_m"], 0.0, 1.0))
frac_line_now = 1.0 - frac_now if reversed_trip else frac_now

now_row = (
    pl.DataFrame({"route_wkb": [route_wkb]})
    .with_columns(route_geom=st.from_wkb("route_wkb"))
    .with_columns(
        pt=pl.col("route_geom").st.interpolate(frac_line_now, normalized=True)
    )
    .with_columns(lon=pl.col("pt").st.x(), lat=pl.col("pt").st.y())
    .row(0, named=True)
)

print(f"At t={elapsed_now:.0f}s ({elapsed_now / 60:.1f} min) after the first stop:")
print(f"  s(t) = {s_now / 1000:.3f} km  ({frac_now * 100:.1f}% of trip distance)")
print(f"  Interpolated position: lon={now_row['lon']:.6f}, lat={now_row['lat']:.6f}")
print(f"  Velocity  s'(t) = {float(v_fn(elapsed_now)):.2f} m/s")
print(f"  Accel     s''(t) = {float(a_fn(elapsed_now)):.3f} m/s^2")

viz_df.st.explore(
    "geom", scatterplot_kwargs={"get_radius": 15, "get_fill_color": "red"}
)

In [ ]:
# Visualization 2.5: Route Geometry, Graph Edges, and Stop Projections
import polars as pl
import polars_st as st
from shapely.geometry import LineString

# 1. Underlay for Routes
routes_wkb = (
    df_routes.filter(pl.col("id").is_in(ROUTES))
    .rename({"geometry": "wkb", "id": "route_id"})
    .with_columns(geom=st.from_wkb("wkb"))
    .select(["route_id", "geom"])
)
m = routes_wkb.st.explore(
    "geom", path_kwargs={"get_color": [160, 160, 160, 180], "width_min_pixels": 3}
)

# 2. Original & Projected Stops
proj_rows = []
for row in df_stops_with_dist.iter_rows(named=True):
    route_id = row["route_id"]
    stop_id = row["stop_id"]
    fraction = row["fraction"]
    lat, lon = stop_coord_map[stop_id]
    proj_rows.append(
        {
            "route_id": route_id,
            "stop_id": stop_id,
            "fraction": fraction,
            "orig_lon": lon,
            "orig_lat": lat,
        }
    )

df_proj = pl.DataFrame(proj_rows).join(routes_wkb, on="route_id")
df_proj = df_proj.with_columns(
    proj_pt=pl.col("geom").st.interpolate(pl.col("fraction"), normalized=True),
    orig_coords=pl.concat_list([pl.col("orig_lon"), pl.col("orig_lat")]),
).with_columns(orig_pt=st.point("orig_coords"))

m.add_layer(
    df_proj.st.explore(
        "orig_pt",
        scatterplot_kwargs={"get_fill_color": [255, 0, 0, 200], "radius_min_pixels": 4},
    )
)
m.add_layer(
    df_proj.st.explore(
        "proj_pt",
        scatterplot_kwargs={"get_fill_color": [0, 255, 0, 200], "radius_min_pixels": 4},
    )
)

# 3. Projection Lines (Original to Projected)
df_proj = df_proj.with_columns(
    proj_lon=pl.col("proj_pt").st.x(), proj_lat=pl.col("proj_pt").st.y()
)
pdf_proj = df_proj.to_pandas()
lines_snap = [
    LineString([(row.orig_lon, row.orig_lat), (row.proj_lon, row.proj_lat)]).wkb
    for _, row in pdf_proj.iterrows()
]

df_snap = pl.DataFrame({"geom": lines_snap}).with_columns(geom=st.from_wkb("geom"))
m.add_layer(
    df_snap.st.explore(
        "geom", path_kwargs={"get_color": [255, 255, 0, 150], "width_min_pixels": 2}
    )
)

# 4. Graph Edges
edges_rows = []
for route_id, g in route_graph.items():
    if route_id not in ROUTES:
        continue
    for u, neighbors in g.items():
        for v, _ in neighbors.items():
            lat1, lon1 = stop_coord_map[u]
            lat2, lon2 = stop_coord_map[v]
            edges_rows.append(LineString([(lon1, lat1), (lon2, lat2)]).wkb)

df_edges = pl.DataFrame({"geom": edges_rows}).with_columns(geom=st.from_wkb("geom"))
m.add_layer(
    df_edges.st.explore(
        "geom", path_kwargs={"get_color": [0, 0, 255, 120], "width_min_pixels": 1}
    )
)

print(
    "Red: Original Stops | Green: Projected Stops | Yellow: Snap Lines | Blue: Graph Edges | Gray: Route Geom"
)
m

In [ ]:
# Visualization 3: animated TripsLayer.
#
# IMPORTANT: TripsLayer only draws data at the layer's `current_time`. You
# must click the Play button on the widget underneath the map to see motion;
# otherwise you're just looking at whichever trains happen to be alive at a
# single frozen instant.
#
# Synchronization: in real wall-clock time each trip has its own start, so at
# any fixed current_time only one or two trains are on screen. To make the
# animation visually informative we time-align every trip to a common epoch
# (SYNC_TRIPS = True). Flip it to False to play in true wall-clock time.
#
# Note: this requires `movingpandas` (added to science/pyproject.toml).
# Run `uv sync` (or `pip install movingpandas`) if the import below fails.

from datetime import datetime, timedelta, timezone

import pandas as pd
import geopandas as gpd
import movingpandas as mpd
import ipywidgets as widgets
from lonboard import Map, TripsLayer, PathLayer

TRIPS_TO_ANIMATE = min(15, len(valid_trips))
N_SAMPLES = 150  # path waypoints per trip
SYNC_TRIPS = True
EPOCH = datetime(2025, 1, 1, tzinfo=timezone.utc)

anim_trips = valid_trips[:TRIPS_TO_ANIMATE]

# Build a long (trip_id, route_id, t, fraction) frame by sampling each trip's
# spline uniformly in time. We un-flip the fraction for reversed trips so it
# indexes the *stored* route line correctly before passing to st.interpolate.
sample_rows: list[dict] = []
for trip in anim_trips:
    t_sec_trip = trip_time_seconds(trip["arrivals"])
    s_m_trip = trip["s_m"]
    spline_trip = fit_cubic(t_sec_trip, s_m_trip, clamped=True)
    times_s = np.linspace(t_sec_trip[0], t_sec_trip[-1], N_SAMPLES)
    s_vals = spline_trip(times_s)
    fracs = np.clip(s_vals / trip["route_length_m"], 0.0, 1.0)
    if trip.get("reversed"):
        fracs = 1.0 - fracs
    t0 = EPOCH if SYNC_TRIPS else trip["arrivals"][0]
    for ts_s, f in zip(times_s, fracs):
        sample_rows.append(
            {
                "trip_id": trip["trip_id"],
                "route_id": trip["route_id"],
                "fraction": float(f),
                "t": t0 + dt.timedelta(seconds=float(ts_s)),
            }
        )

long_df = pl.DataFrame(sample_rows)

# Attach each sample's route geometry and resolve lon/lat via polars-st.
routes_min = df_routes.select(["id", "geometry"]).rename(
    {"id": "route_id", "geometry": "route_wkb"}
)
points_long = (
    long_df.join(routes_min, on="route_id", how="left")
    .with_columns(route_geom=st.from_wkb("route_wkb"))
    .with_columns(
        pt=pl.col("route_geom").st.interpolate(pl.col("fraction"), normalized=True),
    )
    .with_columns(
        lon=pl.col("pt").st.x(),
        lat=pl.col("pt").st.y(),
    )
    .select(["trip_id", "route_id", "t", "lon", "lat"])
    .sort(["trip_id", "t"])
)

# movingpandas wants a GeoDataFrame indexed by timestamp with one row per
# (trajectory, sample) and a traj-id column.
pdf = points_long.to_pandas()
pdf["t"] = pd.to_datetime(pdf["t"], utc=True)
gdf = gpd.GeoDataFrame(
    pdf,
    geometry=gpd.points_from_xy(pdf["lon"], pdf["lat"]),
    crs="EPSG:4326",
).set_index("t")

traj_collection = mpd.TrajectoryCollection(gdf, traj_id_col="trip_id")

t_min_idx, t_max_idx = pdf.index.min(), pdf.index.max()
print(
    f"Built TrajectoryCollection with {len(traj_collection.trajectories)} trajectories "
    f"({N_SAMPLES} waypoints each)"
)
print(f"Animated time range: {t_min_idx}  ->  {t_max_idx}")
print(f"  Duration: {(t_max_idx - t_min_idx)}")
print(
    "To see motion, click the triangular Play button on the widget below the "
    "map. TripsLayer only draws data at its current_time -- without pressing "
    "Play, time never advances."
)

# Color per route (MTA-ish: 4 = green, 3 = red).
route_rgb = {"4": [0, 147, 60], "3": [238, 53, 46]}
traj_colors = np.array(
    [
        route_rgb.get(traj.df["route_id"].iloc[0], [128, 128, 128])
        for traj in traj_collection.trajectories
    ],
    dtype=np.uint8,
)

trips_layer = TripsLayer.from_movingpandas(
    traj_collection,
    get_color=traj_colors,
    width_min_pixels=10,  # chunkier trains so they're easy to spot
    trail_length=360,  # seconds of fading trail behind each train
    cap_rounded=True,
    joint_rounded=True,
    opacity=0.95,
    fade_trail=True,
)

# Thin static underlay of the route lines, so you can see the corridor even
# between trips.
static_routes = (
    df_routes.filter(pl.col("id").is_in([t["route_id"] for t in anim_trips]))
    .rename({"geometry": "wkb", "id": "route_id"})
    .with_columns(geom=st.from_wkb("wkb"))
    .select(["route_id", "geom"])
    .to_pandas()
)
static_gdf = gpd.GeoDataFrame(
    {"route_id": static_routes["route_id"]},
    geometry=gpd.GeoSeries.from_wkb(static_routes["geom"]),
    crs="EPSG:4326",
)
underlay = PathLayer.from_geopandas(
    static_gdf,
    get_color=[160, 160, 160, 180],
    width_min_pixels=3,
)

m = Map([underlay, trips_layer], height=650)

# Advance 10 "data seconds" per animation frame at 30 fps, i.e. 300x real time.
controller = trips_layer.animate(
    step=timedelta(seconds=10), fps=30, strftime_fmt="%H:%M:%S"
)

# Bundle map + controller in a VBox so the Play button is directly below the map.
widgets.VBox([m, controller])